# IEEE-CIS Fraud Detection - Team 2 Member 3 XGBoost + CatBoost Baseline

Colab-first baselines on merged Team 1 parquet outputs.

- Chronological train/valid split using `TransactionDT`
- XGBoost: median-imputed numerics + frequency-bucketed categorical integer codes
- CatBoost: native categorical handling with missing strings filled as `__MISSING__`
- Save validation/test predictions, metrics, and feature importance to Google Drive

In [1]:
from pathlib import Path

RANDOM_SEED = 42
TARGET_COLUMN = "isFraud"
ID_COLUMN = "TransactionID"
VALID_SIZE = 0.2
SMOKE_TEST = False
SMOKE_ROWS = 100_000
MAX_CAT_FOR_XGB = 256

DATA_ROOT = Path("/content/drive/MyDrive/ieee_cis_fraud")
MERGED_DIR = DATA_ROOT / "merged"
OUTPUT_DIR = DATA_ROOT / "member3_xgb_catboost_baseline"
TRAIN_PATH = MERGED_DIR / "train_member1_member2_merged.parquet"
TEST_PATH = MERGED_DIR / "test_member1_member2_merged.parquet"

METRICS_PATH = OUTPUT_DIR / "member3_xgb_catboost_metrics.json"
VALID_PREDICTIONS_PATH = OUTPUT_DIR / "member3_xgb_catboost_validation_predictions.parquet"
TEST_PREDICTIONS_PATH = OUTPUT_DIR / "member3_xgb_catboost_test_predictions.parquet"
XGB_IMPORTANCE_PATH = OUTPUT_DIR / "member3_xgb_feature_importance.csv"
CAT_IMPORTANCE_PATH = OUTPUT_DIR / "member3_catboost_feature_importance.csv"
XGB_MODEL_PATH = OUTPUT_DIR / "member3_xgb_model.json"
CAT_MODEL_PATH = OUTPUT_DIR / "member3_catboost_model.cbm"

TRAIN_PATH, TEST_PATH, OUTPUT_DIR

(PosixPath('/content/drive/MyDrive/ieee_cis_fraud/merged/train_member1_member2_merged.parquet'),
 PosixPath('/content/drive/MyDrive/ieee_cis_fraud/merged/test_member1_member2_merged.parquet'),
 PosixPath('/content/drive/MyDrive/ieee_cis_fraud/member3_xgb_catboost_baseline'))

In [2]:
!pip -q install xgboost catboost pyarrow pandas scikit-learn torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.9 MB/s eta 0:00:00


In [3]:
from google.colab import drive

drive.mount("/content/drive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

Mounted at /content/drive
TRAIN_PATH: /content/drive/MyDrive/ieee_cis_fraud/merged/train_member1_member2_merged.parquet
TEST_PATH: /content/drive/MyDrive/ieee_cis_fraud/merged/test_member1_member2_merged.parquet
OUTPUT_DIR: /content/drive/MyDrive/ieee_cis_fraud/member3_xgb_catboost_baseline


In [4]:
import gc
import json

import numpy as np
import pandas as pd
import torch
from catboost import CatBoostClassifier, CatBoostError, Pool
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier


def memory_usage_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / 1024**2


def downcast_numeric_frame(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    for col in result.select_dtypes(include=["int", "int64", "int32", "int16"]).columns:
        result[col] = pd.to_numeric(result[col], downcast="integer")
    for col in result.select_dtypes(include=["float", "float64", "float32"]).columns:
        result[col] = pd.to_numeric(result[col], downcast="float")
    return result


def detect_categorical_columns(*frames: pd.DataFrame) -> list[str]:
    cols = []
    for col in frames[0].columns:
        if any(frame[col].dtype == object or str(frame[col].dtype).startswith("string") for frame in frames if col in frame):
            cols.append(col)
    return cols


def fit_xgb_preprocessor(X_train: pd.DataFrame, categorical_cols: list[str], max_categories: int):
    numeric_cols = [c for c in X_train.columns if c not in categorical_cols]
    imputer = SimpleImputer(strategy="median")
    imputer.fit(X_train[numeric_cols].astype("float64"))

    mappings = {}
    for col in categorical_cols:
        values = X_train[col].astype("string").fillna("__MISSING__")
        top_values = values.value_counts().head(max_categories - 1).index.tolist()
        mappings[col] = {str(value): idx + 1 for idx, value in enumerate(top_values)}  # 0 = other/unseen

    return numeric_cols, imputer, mappings


def transform_xgb(X: pd.DataFrame, numeric_cols: list[str], categorical_cols: list[str], imputer: SimpleImputer, mappings: dict[str, dict[str, int]]) -> pd.DataFrame:
    num = imputer.transform(X[numeric_cols].astype("float64")).astype(np.float32)
    num_df = pd.DataFrame(num, columns=numeric_cols, index=X.index)

    cat_parts = []
    for col in categorical_cols:
        encoded = (
            X[col]
            .astype("string")
            .fillna("__MISSING__")
            .map(lambda value: mappings[col].get(str(value), 0))
            .astype("int16")
        )
        cat_parts.append(encoded.rename(col))

    if cat_parts:
        return pd.concat([num_df, pd.concat(cat_parts, axis=1)], axis=1)
    return num_df


def prepare_catboost_frame(X: pd.DataFrame, categorical_cols: list[str]) -> pd.DataFrame:
    result = X.copy()
    for col in categorical_cols:
        result[col] = result[col].astype("string").fillna("__MISSING__").astype(str)
    return result


print("Torch CUDA available:", torch.cuda.is_available())

Torch CUDA available: True


In [7]:
assert TRAIN_PATH.exists(), f"Missing train parquet: {TRAIN_PATH}"
# assert TEST_PATH.exists(), f"Missing test parquet: {TEST_PATH}"

train_df = pd.read_parquet(TRAIN_PATH)
# test_df = pd.read_parquet(TEST_PATH)

if SMOKE_TEST:
    train_df = train_df.sample(n=min(SMOKE_ROWS, len(train_df)), random_state=RANDOM_SEED).reset_index(drop=True)
    # test_df = test_df.sample(n=min(SMOKE_ROWS, len(test_df)), random_state=RANDOM_SEED).reset_index(drop=True)

train_df = downcast_numeric_frame(train_df)
# test_df = downcast_numeric_frame(test_df)

print("train:", train_df.shape, "MB:", round(memory_usage_mb(train_df), 2))
# print("test:", test_df.shape, "MB:", round(memory_usage_mb(test_df), 2))

train: (590540, 151) MB: 1094.28


In [8]:
assert TARGET_COLUMN in train_df.columns
# assert TARGET_COLUMN not in test_df.columns
# assert ID_COLUMN in train_df.columns and ID_COLUMN in test_df.columns
assert "TransactionDT" in train_df.columns

train_shape_full = train_df.shape
# test_shape_full = test_df.shape

y = train_df[TARGET_COLUMN].astype(np.int64)
train_ids = train_df[ID_COLUMN].copy()
# test_ids = test_df[ID_COLUMN].copy()

X = train_df.drop(columns=[TARGET_COLUMN, ID_COLUMN])
# X_test_raw = test_df.drop(columns=[ID_COLUMN])

sort_idx = X["TransactionDT"].sort_values(kind="mergesort").index
X = X.loc[sort_idx].reset_index(drop=True)
y = y.loc[sort_idx].reset_index(drop=True)
train_ids = train_ids.loc[sort_idx].reset_index(drop=True)

split_idx = int(len(X) * (1 - VALID_SIZE))
X_train_raw = X.iloc[:split_idx].copy()
X_valid_raw = X.iloc[split_idx:].copy()
y_train = y.iloc[:split_idx].to_numpy(dtype=np.int64)
y_valid = y.iloc[split_idx:].to_numpy(dtype=np.int64)
valid_ids = train_ids.iloc[split_idx:].reset_index(drop=True)

# Detect categories without X_test_raw
categorical_cols = detect_categorical_columns(X_train_raw, X_valid_raw)
numeric_cols = [c for c in X_train_raw.columns if c not in categorical_cols]

print("X_train:", X_train_raw.shape, "X_valid:", X_valid_raw.shape)
print("Fraud rate train/valid:", round(y_train.mean(), 6), round(y_valid.mean(), 6))
print("Categorical columns:", len(categorical_cols), categorical_cols[:10])
print("Numeric columns:", len(numeric_cols))

del train_df, X, y, train_ids # Removed test_df
gc.collect()

X_train: (472432, 149) X_valid: (118108, 149)
Fraud rate train/valid: 0.035135 0.034409
Categorical columns: 31 ['card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6']
Numeric columns: 118


313

In [9]:
numeric_cols_xgb, xgb_imputer, xgb_cat_maps = fit_xgb_preprocessor(
    X_train_raw,
    categorical_cols,
    max_categories=MAX_CAT_FOR_XGB,
)

X_train_xgb = transform_xgb(X_train_raw, numeric_cols_xgb, categorical_cols, xgb_imputer, xgb_cat_maps)
X_valid_xgb = transform_xgb(X_valid_raw, numeric_cols_xgb, categorical_cols, xgb_imputer, xgb_cat_maps)
# X_test_xgb = transform_xgb(X_test_raw, numeric_cols_xgb, categorical_cols, xgb_imputer, xgb_cat_maps)

assert np.isfinite(X_train_xgb.to_numpy()).all()
assert np.isfinite(X_valid_xgb.to_numpy()).all()
# assert np.isfinite(X_test_xgb.to_numpy()).all()

xgb_params = dict(
    objective="binary:logistic",      # Binary classification; output is fraud probability.
    eval_metric="auc",               # Track ROC-AUC on validation data.
    n_estimators=2000,                # Maximum boosting rounds; early stopping usually stops earlier.
    learning_rate=0.03,               # Smaller step size per tree for steadier learning.
    max_depth=6,                      # Maximum depth of each tree; controls tree complexity.
    min_child_weight=10,              # Minimum child-node weight; larger values reduce overfitting.
    subsample=0.8,                    # Use 80% of rows per tree to regularize.
    colsample_bytree=0.8,             # Use 80% of features per tree to regularize.
    reg_alpha=0.1,                    # L1 regularization on leaf weights.
    reg_lambda=1.0,                   # L2 regularization on leaf weights.
    random_state=RANDOM_SEED,         # Reproducible training.
    n_jobs=-1,                        # Use all available CPU cores when CPU work is needed.
    early_stopping_rounds=100,        # Stop if validation AUC does not improve for 100 rounds.
)

try:
    print("Training XGBoost on GPU...")
    xgb_model = XGBClassifier(**xgb_params, tree_method="hist", device="cuda")
    xgb_model.fit(X_train_xgb, y_train, eval_set=[(X_valid_xgb, y_valid)], verbose=100)
    xgb_device = "cuda"
except Exception as exc:
    print("XGBoost GPU failed; falling back to CPU:", exc)
    xgb_model = XGBClassifier(**xgb_params, tree_method="hist")
    xgb_model.fit(X_train_xgb, y_train, eval_set=[(X_valid_xgb, y_valid)], verbose=100)
    xgb_device = "cpu"

xgb_valid_probs = xgb_model.predict_proba(X_valid_xgb)[:, 1]
# xgb_test_probs = xgb_model.predict_proba(X_test_xgb)[:, 1]
xgb_auc = roc_auc_score(y_valid, xgb_valid_probs)
print("XGBoost valid ROC-AUC:", round(float(xgb_auc), 6), "device:", xgb_device)

xgb_model.save_model(str(XGB_MODEL_PATH))

xgb_importance = pd.DataFrame({
    "feature": X_train_xgb.columns,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending=False)
xgb_importance.to_csv(XGB_IMPORTANCE_PATH, index=False)

# Keep predictions; free matrices before CatBoost.
del X_train_xgb, X_valid_xgb # Removed X_test_xgb
gc.collect()

Training XGBoost on GPU...
[0]	validation_0-auc:0.79450
[100]	validation_0-auc:0.87621
[200]	validation_0-auc:0.88884
[300]	validation_0-auc:0.89528
[400]	validation_0-auc:0.89884
[500]	validation_0-auc:0.90180
[600]	validation_0-auc:0.90423
[700]	validation_0-auc:0.90604
[800]	validation_0-auc:0.90757
[900]	validation_0-auc:0.90930
[1000]	validation_0-auc:0.91053
[1100]	validation_0-auc:0.91167
[1200]	validation_0-auc:0.91251
[1300]	validation_0-auc:0.91305
[1400]	validation_0-auc:0.91338
[1500]	validation_0-auc:0.91400
[1600]	validation_0-auc:0.91421
[1700]	validation_0-auc:0.91494
[1800]	validation_0-auc:0.91495
[1802]	validation_0-auc:0.91494


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [13:21:49] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


XGBoost valid ROC-AUC: 0.914966 device: cuda


0

In [10]:
X_train_cat = prepare_catboost_frame(X_train_raw, categorical_cols)
X_valid_cat = prepare_catboost_frame(X_valid_raw, categorical_cols)
# X_test_cat = prepare_catboost_frame(X_test_raw, categorical_cols)

cat_features = categorical_cols
train_pool = Pool(X_train_cat, y_train, cat_features=cat_features)
valid_pool = Pool(X_valid_cat, y_valid, cat_features=cat_features)
# test_pool = Pool(X_test_cat, cat_features=cat_features)

cat_params = dict(
    loss_function="Logloss",          # Binary classification loss.
    eval_metric="AUC",               # Track ROC-AUC on validation data.
    iterations=3000,                  # Maximum boosting rounds; early stopping usually stops earlier.
    learning_rate=0.03,               # Smaller step size per tree for steadier learning.
    depth=8,                          # Tree depth; higher can model interactions but may overfit.
    l2_leaf_reg=3.0,                  # L2 regularization on leaf values.
    random_seed=RANDOM_SEED,          # Reproducible training.
    auto_class_weights="Balanced",   # Adjust class weights for fraud imbalance.
    allow_writing_files=False,        # Avoid CatBoost temp/log files in the working directory.
    verbose=100,                      # Print training progress every 100 iterations.
)

try:
    print("Training CatBoost on GPU...")
    cat_model = CatBoostClassifier(**cat_params, task_type="GPU", devices="0")
    cat_model.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=150)
    cat_device = "gpu"
except (CatBoostError, RuntimeError) as exc:
    print("CatBoost GPU failed; falling back to CPU:", exc)
    cat_model = CatBoostClassifier(**cat_params, task_type="CPU")
    cat_model.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=150)
    cat_device = "cpu"

cat_valid_probs = cat_model.predict_proba(valid_pool)[:, 1]
# cat_test_probs = cat_model.predict_proba(test_pool)[:, 1]
cat_auc = roc_auc_score(y_valid, cat_valid_probs)
print("CatBoost valid ROC-AUC:", round(float(cat_auc), 6), "device:", cat_device)

cat_model.save_model(str(CAT_MODEL_PATH))
cat_importance = pd.DataFrame({
    "feature": X_train_cat.columns,
    "importance": cat_model.get_feature_importance(train_pool),
}).sort_values("importance", ascending=False)
cat_importance.to_csv(CAT_IMPORTANCE_PATH, index=False)

del X_train_cat, X_valid_cat, train_pool, valid_pool # Removed test_pool and X_test_cat
gc.collect()

Training CatBoost on GPU...


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8209940	best: 0.8209940 (0)	total: 275ms	remaining: 13m 45s
100:	test: 0.8748108	best: 0.8748108 (100)	total: 22.1s	remaining: 10m 34s
200:	test: 0.8895956	best: 0.8895956 (200)	total: 43.8s	remaining: 10m 10s
300:	test: 0.8970032	best: 0.8970032 (300)	total: 1m 5s	remaining: 9m 49s
400:	test: 0.9002761	best: 0.9002960 (391)	total: 1m 27s	remaining: 9m 28s
500:	test: 0.9037451	best: 0.9037451 (500)	total: 1m 49s	remaining: 9m 5s
600:	test: 0.9063020	best: 0.9063020 (600)	total: 2m 11s	remaining: 8m 46s
700:	test: 0.9086074	best: 0.9086163 (699)	total: 2m 33s	remaining: 8m 23s
800:	test: 0.9093963	best: 0.9093963 (800)	total: 2m 55s	remaining: 8m 3s
900:	test: 0.9098737	best: 0.9098737 (900)	total: 3m 17s	remaining: 7m 41s
1000:	test: 0.9097391	best: 0.9100557 (934)	total: 3m 40s	remaining: 7m 19s
bestTest = 0.910055697
bestIteration = 934
Shrink model to first 935 iterations.
CatBoost valid ROC-AUC: 0.910056 device: gpu


0

In [11]:
valid_predictions = pd.DataFrame({
    ID_COLUMN: valid_ids.astype(np.int64),
    "y_true": y_valid.astype(np.int8),
    "xgb_prob_fraud": xgb_valid_probs.astype(np.float32),
    "catboost_prob_fraud": cat_valid_probs.astype(np.float32),
})
valid_predictions.to_parquet(VALID_PREDICTIONS_PATH, index=False)

# test_predictions = pd.DataFrame({
#     ID_COLUMN: test_ids.astype(np.int64),
#     "xgb_prob_fraud": xgb_test_probs.astype(np.float32),
#     "catboost_prob_fraud": cat_test_probs.astype(np.float32),
# })
# test_predictions.to_parquet(TEST_PREDICTIONS_PATH, index=False)

metrics = {
    "valid_auc": {
        "xgboost": float(xgb_auc),
        "catboost": float(cat_auc),
    },
    "device": {
        "xgboost": xgb_device,
        "catboost": cat_device,
    },
    "best_iteration": {
        "xgboost": int(getattr(xgb_model, "best_iteration", -1) if getattr(xgb_model, "best_iteration", None) is not None else -1),
        "catboost": int(cat_model.get_best_iteration() or -1),
    },
    "train_shape_full": list(train_shape_full),
    # "test_shape_full": list(test_shape_full),
    "train_rows_used": int(len(y_train)),
    "valid_rows_used": int(len(y_valid)),
    "categorical_columns": categorical_cols,
    "numeric_column_count": len(numeric_cols),
    "max_cat_for_xgb": MAX_CAT_FOR_XGB,
    "outputs": {
        "valid_predictions": str(VALID_PREDICTIONS_PATH),
        # "test_predictions": str(TEST_PREDICTIONS_PATH),
        "xgb_model": str(XGB_MODEL_PATH),
        "catboost_model": str(CAT_MODEL_PATH),
        "xgb_importance": str(XGB_IMPORTANCE_PATH),
        "catboost_importance": str(CAT_IMPORTANCE_PATH),
    },
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print("Wrote metrics:", METRICS_PATH)
print("Wrote validation predictions:", VALID_PREDICTIONS_PATH)
# print("Wrote test predictions:", TEST_PREDICTIONS_PATH)
print("AUC summary:", metrics["valid_auc"])

Wrote metrics: /content/drive/MyDrive/ieee_cis_fraud/member3_xgb_catboost_baseline/member3_xgb_catboost_metrics.json
Wrote validation predictions: /content/drive/MyDrive/ieee_cis_fraud/member3_xgb_catboost_baseline/member3_xgb_catboost_validation_predictions.parquet
AUC summary: {'xgboost': 0.9149659104670749, 'catboost': 0.9100557688122584}
